# Laboratório PySpark: O Caos na Prática (OOM e Data Skew)

Neste notebook, vamos forçar o Spark ao extremo para que o **Data Skew** fique absurdamente visível na Spark UI e os Jobs demorem minutos (ou quebrem).

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import count, col, row_number
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Lab: Bad Practices OOM") \
    .getOrCreate()

spark

### 1. Leitura sem Schema (Small Files Problem)

In [5]:
ENVIRONMENT = "databricks_volume" # Altere para "local" quando rodar no Docker

if ENVIRONMENT == "local":
    BASE_PATH = "file:///home/jovyan/work/data"
elif ENVIRONMENT == "databricks_volume":
    # Novo caminho usando Unity Catalog Volumes (muito mais moderno que o antigo DBFS)
    BASE_PATH = "/Volumes/workspace/default/raw_data"

# Se vc fizer o upload dos CSVs soltos direto no volume, o raw_path é o próprio BASE_PATH
raw_path = f"{BASE_PATH}/"
silver_path = f"{BASE_PATH}/silver/churn_optimized.parquet"

print(f"[Config] Rodando no ambiente: {ENVIRONMENT}")
print(f"[Config] BASE_PATH = {BASE_PATH}\n")

# Lendo o CSV bruto sem inferSchema
df_bad = spark.read.csv(raw_path, header=True, inferSchema=True)
print(f"Linhas lidas: {df_bad.count()}")


Linhas lidas: 3521500


### 2. O VERDADEIRO GARGALO: Window Function em cima do Skew
Vamos fazer uma operação de Janela (Window). O Spark é OBRIGADO a enviar todas as linhas que possuem o mesmo `customerID` para a **mesma CPU** e ordená-las na memória antes de processar.

Como criamos ~2.8 milhões de linhas com o mesmo `customerID` ('SK-999999-SKEW'), uma única CPU (Task) vai tentar alocar e ordenar esses milhões de registros sozinha, enquanto as outras Tasks processam apenas 1 linha. Isso fará o Job demorar vários minutos na Spark UI e provavelmente vazar dados para o disco (Spill) ou estourar a memória (OOM).

In [6]:
# Particionamos pelo customerID e ordenamos pelos encargos mensais.
window_spec = Window.partitionBy("customerID").orderBy("MonthlyCharges")

df_ranked = df_bad.withColumn("rank", row_number().over(window_spec))

print("Iniciando a Ação. ISSO VAI DEMORAR!")
print("CORRA PARA A SPARK UI (http://localhost:4040), ABA 'STAGES' E VEJA A TASK ASSASSINA.")

# Executa a ação pesada
df_ranked.count()

Iniciando a Ação. ISSO VAI DEMORAR!
CORRA PARA A SPARK UI (http://localhost:4040), ABA 'STAGES' E VEJA A TASK ASSASSINA.


3521500

### 3. Cross Join da Morte (Para garantir o OOM, se o passo anterior não matou o cluster)
Se o `row_number` ali em cima não demorou 3 minutos na sua máquina (por ela ser muito rápida), descomente a célula abaixo.
Faremos um self-join desprotegido na chave desbalanceada. Ele tentará criar Bilhões de combinações.

In [8]:
# DESCOMENTE AS LINHAS ABAIXO POR SUA CONTA E RISCO!
# print("Iniciando o Self-Join do Caos...")
# df_join = df_bad.join(df_bad.select("customerID", "SeniorCitizen"), on="customerID", how="inner")
# df_join.count()